# 00. Data Collection Readiness

이 노트북은 `ROOT_PATH/data collection/` 아래 수집 산출물이 data-analysis 01~05번에서 필요한 입력 계약을 만족하는지 점검합니다. 자동/수동 수집 산출물을 `ROOT_PATH/data/`로 복사하지 않습니다.

없는 수동 수집 데이터는 `z_pa_` 접두어 폴더만 생성합니다.

In [ ]:
# ============================================================
# 00 공통 경로 설정
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('[INFO] Google Colab 환경이 아니므로 drive.mount를 건너뜁니다.')

from pathlib import Path
import os
import pandas as pd

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_COLLECTION_PATH = os.path.join(ROOT_PATH, "data collection") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

ROOT_DIR = Path(ROOT_PATH)
DATA_COLLECTION_DIR = Path(DATA_COLLECTION_PATH)
MANIFEST_DIR = DATA_COLLECTION_DIR / "_manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)

print(f"ROOT_PATH           = {ROOT_PATH}")
print(f"DATA_COLLECTION_PATH= {DATA_COLLECTION_PATH}")
print(f"PROCESSED_PATH      = {PROCESSED_PATH}")
print(f"MANIFEST_DIR        = {MANIFEST_DIR}")

In [ ]:
# ============================================================
# 수집 산출물 탐색/점검 유틸
# ============================================================
def latest_under(base, pattern):
    base = Path(base)
    if not base.exists():
        return None
    files = [p for p in base.glob(pattern) if p.exists() and p.is_file()]
    if not files:
        return None
    return sorted(files, key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)[0]


def first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None


def read_columns(path):
    path = Path(path)
    try:
        if path.suffix.lower() in [".csv", ".txt"]:
            for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
                try:
                    return list(pd.read_csv(path, encoding=enc, nrows=0).columns)
                except Exception:
                    pass
        if path.suffix.lower() in [".xls", ".xlsx", ".html", ".htm"]:
            try:
                return list(pd.read_excel(path, nrows=0).columns)
            except Exception:
                tables = pd.read_html(path, header=0)
                return list(tables[0].columns) if tables else []
    except Exception as e:
        return [f"COLUMN_READ_ERROR: {e}"]
    return []


def ensure_manual_dataset_folders(dataset_name):
    base = DATA_COLLECTION_DIR / dataset_name
    for sub in ["raw", "final", "logs"]:
        (base / sub).mkdir(parents=True, exist_ok=True)
    return base

In [ ]:
# ============================================================
# data-analysis 01~05 입력 계약 점검
# ============================================================
SPECS = [
    {"name": "crude", "dataset": "crude", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "crude" / "final", "crude_*.csv"),
     "expected": DATA_COLLECTION_DIR / "crude" / "final" / "crude_*.csv"},
    {"name": "retail_avg", "dataset": "retail_avg", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "retail_avg" / "final", "retail_avg_*.csv"),
     "expected": DATA_COLLECTION_DIR / "retail_avg" / "final" / "retail_avg_*.csv"},
    {"name": "brand_gasoline", "dataset": "brand_price", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "brand_price" / "final", "brand_gasoline_*.csv"),
     "expected": DATA_COLLECTION_DIR / "brand_price" / "final" / "brand_gasoline_*.csv"},
    {"name": "brand_diesel", "dataset": "brand_price", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "brand_price" / "final", "brand_diesel_*.csv"),
     "expected": DATA_COLLECTION_DIR / "brand_price" / "final" / "brand_diesel_*.csv"},
    {"name": "fx_usdkrw", "dataset": "fx_usdkrw", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "fx_usdkrw" / "final", "fx_usdkrw_*.csv"),
     "expected": DATA_COLLECTION_DIR / "fx_usdkrw" / "final" / "fx_usdkrw_*.csv"},
    {"name": "intl_products", "dataset": "intl_products", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "intl_products" / "final", "intl_products_*.csv"),
     "expected": DATA_COLLECTION_DIR / "intl_products" / "final" / "intl_products_*.csv"},
    {"name": "intl_product_diesel_0001", "dataset": "intl_products", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "intl_products" / "final", "intl_product_diesel(0.001)_*.csv"),
     "expected": DATA_COLLECTION_DIR / "intl_products" / "final" / "intl_product_diesel(0.001)_*.csv"},
    {"name": "gasoline_tax_trend", "dataset": "fuel_tax_trend", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "fuel_tax_trend" / "final", "gasoline_tax_trend_*.xls"),
     "expected": DATA_COLLECTION_DIR / "fuel_tax_trend" / "final" / "gasoline_tax_trend_*.xls"},
    {"name": "diesel_tax_trend", "dataset": "fuel_tax_trend", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "fuel_tax_trend" / "final", "diesel_tax_trend_*.xls"),
     "expected": DATA_COLLECTION_DIR / "fuel_tax_trend" / "final" / "diesel_tax_trend_*.xls"},
    {"name": "refinery_weekly_supply", "dataset": "refinery_weekly_supply", "required_for": "01", "required": True,
     "src": latest_under(DATA_COLLECTION_DIR / "refinery_weekly_supply" / "final", "refinery_weekly_supply_prices_by_product_*.csv"),
     "expected": DATA_COLLECTION_DIR / "refinery_weekly_supply" / "final" / "refinery_weekly_supply_prices_by_product_*.csv"},
    {"name": "korea_fuel_tax_price_policies", "dataset": "z_pa_policy", "required_for": "05", "required": True, "manual_dataset": "z_pa_policy",
     "src": DATA_COLLECTION_DIR / "z_pa_policy" / "final" / "korea_fuel_tax_price_policies.csv",
     "expected": DATA_COLLECTION_DIR / "z_pa_policy" / "final" / "korea_fuel_tax_price_policies.csv"},
    {"name": "official_land_price", "dataset": "official_land_price", "required_for": "not_used_by_data_analysis_01_05", "required": False,
     "src": latest_under(DATA_COLLECTION_DIR / "official_land_price" / "final", "*.csv"),
     "expected": DATA_COLLECTION_DIR / "official_land_price" / "final" / "*.csv"},
    {"name": "facility_data", "dataset": "z_pa_facility", "required_for": "not_used_by_data_analysis_01_05", "required": False,
     "src": DATA_COLLECTION_DIR / "z_pa_facility" / "final" / "facility_data.csv",
     "expected": DATA_COLLECTION_DIR / "z_pa_facility" / "final" / "facility_data.csv"},
]

manifest_rows = []
for spec in SPECS:
    src = spec["src"]
    src_exists = bool(src is not None and Path(src).exists())
    columns = []
    manual_folder = ""

    if src_exists:
        status = "available"
        columns = read_columns(src)
    elif spec.get("manual_dataset"):
        manual_base = ensure_manual_dataset_folders(spec["manual_dataset"])
        manual_folder = str(manual_base)
        status = "missing_manual_folder_created"
    elif spec["required"]:
        status = "missing_collected_output"
    else:
        status = "missing_optional_not_used_by_01_05"

    manifest_rows.append({
        "name": spec["name"],
        "dataset": spec["dataset"],
        "required_for": spec["required_for"],
        "required": spec["required"],
        "status": status,
        "source_path": str(src) if src is not None else "",
        "expected_path_or_pattern": str(spec["expected"]),
        "manual_folder": manual_folder,
        "columns": ", ".join(map(str, columns)),
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "data_analysis_input_manifest.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

print(f"[SAVE] {manifest_path}")
display(manifest) if "display" in globals() else print(manifest.to_string(index=False))

missing_required = manifest[(manifest["required"]) & (manifest["status"].str.startswith("missing"))]
if len(missing_required) > 0:
    print("\n[필수 입력 중 없는 데이터]")
    print(missing_required[["name", "dataset", "required_for", "expected_path_or_pattern", "manual_folder"]].to_string(index=False))
    print("\n수동 수집 대상은 manual_folder의 final/ 아래에 파일을 넣고 다시 실행하세요. 자동 수집 대상이 비어 있으면 해당 수집 코드를 먼저 실행해야 합니다.")
else:
    print("\n[OK] data-analysis 01~05번에서 필요한 수집 산출물이 준비되어 있습니다.")

print("\n[DONE] 00번 수집 산출물 점검 완료")